# Multi-Host Distributed TorchTitan Training on GKE with Shared Filestore (`ReadWriteMany`) Volume

This notebook demonstrates how to scale **TorchTitan** (`torchtitan.experiments.tpu.train`) from interactive single-host prototyping to **multi-host TPU slices on GKE** (e.g. `tpu-v5-8-multi-host` with 2 hosts × 4 chips = 8 TPU chips, or `tpu-v5-32-multi-host` with 8 hosts × 4 chips = 32 TPU chips) using **Kubeflow Trainer (`TrainJob`)** and a shared **`ReadWriteMany` (RWX) Filestore volume** (`shared-workspace-rwx`).

### How Workspaces + Shared RWX + Kubeflow Trainer Simplifies Multi-Host TorchTitan

| Step | Legacy Multi-Host GKE Process (`torchtitan/experiments/tpu/README.md`) | GKE Workspaces + `jupyter-workspace-sync` + Kubeflow Trainer |
| :--- | :--- | :--- |
| **1. Cluster & Tooling** | Install `gcluster` (Cluster Toolkit), Terraform, and local Docker; deploy a separate cluster blueprint | **Same GKE Workspace & VS Code window** used for single-host prototyping; GKE `ComputeClass` auto-provisions multi-host TPU slices on demand |
| **2. Code & Dependency Packaging** | Copy `torch_tpu-*.whl`, run `docker build -f torchtitan/experiments/tpu/Dockerfile .`, and `docker push` to Artifact Registry on **every code edit** (**~10–20 min/iteration**) | **Zero Docker builds or pushes**: all TPU worker pods reuse `jupyterlab:latest-tpu` and mount `/home/jovyan/shared/torchtitan` + `/home/jovyan/shared/.pydeps` directly from the shared RWX PVC (**~300 ms local-to-cluster sync**) |
| **3. Tokenizer & Dataset Assets** | Manually run `gcloud storage cp --recursive ./assets/hf gs://...` and configure GCSFUSE mounts | **Zero manual uploads**: tokenizers, configs, and datasets inside `/home/jovyan/shared/torchtitan` are immediately mounted at `/workspace` on every TPU host pod |
| **4. Multi-Host `torchrun` Launch** | Construct 25-line `gcluster job submit` CLI command with manual `WORKERS_0_HOSTNAME` DNS and rendezvous flags | **1 Python function call** (`submit_multihost_torchtitan()`); Kubeflow Trainer's `torch-distributed` runtime and TorchTitan's `_maybe_init_distributed_on_gke()` auto-configure rendezvous and `TORCH_TPU_SLICEBUILDER_ADDRESSES` |

## 1. Verify Shared `ReadWriteMany` PVC & Synced TorchTitan Repository

Assuming you cloned `google-pytorch` (containing `torchtitan`) locally and synced your project to the Workspace via `jupyter-sync`:
1. Locate the synced `torchtitan` repository on `/home/jovyan/shared`.
2. Ensure TorchTitan's requirements (`torchtitan/experiments/tpu/requirements.txt`, `requirements.txt`, `.ci/docker/requirements-flux.txt`) are installed into `/home/jovyan/shared/.pydeps` (`PYTHONUSERBASE`).
3. Verify the `shared-workspace-rwx` `ReadWriteMany` PVC `subPath` mappings via the generic `tpu_trainer.resolve_shared_pvc_subpath()` utility.

In [ ]:
import importlib
import importlib.util
import os
import pathlib
import subprocess
import sys
from kubernetes import client as k8s_client, config as k8s_config

for candidate in [pathlib.Path.cwd(), pathlib.Path("/home/jovyan/shared/gke-workspaces/examples/torchtitan")]:
    if (candidate / "tpu_trainer.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import tpu_trainer

importlib.reload(tpu_trainer)

try:
    k8s_config.load_incluster_config()
except Exception:
    k8s_config.load_kube_config()

namespace = tpu_trainer.get_current_namespace()
shared_pvc_name = os.environ.get("SHARED_RWX_PVC", "shared-workspace-rwx")
shared_mount_path = pathlib.Path(os.environ.get("SHARED_RWX_MOUNT", "/home/jovyan/shared"))

core_v1 = k8s_client.CoreV1Api()
pvc = core_v1.read_namespaced_persistent_volume_claim(name=shared_pvc_name, namespace=namespace)

# 1. Locate the synced TorchTitan repository
search_paths = [
    pathlib.Path(os.environ["TORCHTITAN_DIR"]) if "TORCHTITAN_DIR" in os.environ else None,
    pathlib.Path.cwd() / "torchtitan",
    pathlib.Path.cwd().parent / "torchtitan",
    shared_mount_path / "google-pytorch" / "torchtitan",
    pathlib.Path("/home/jovyan/google-pytorch/torchtitan"),
    shared_mount_path / "torchtitan",
    pathlib.Path("/home/jovyan/torchtitan"),
]
torchtitan_dir = next(
    (p.resolve() for p in search_paths if p and (p / "torchtitan/experiments/tpu/train.py").exists()),
    None,
)
if torchtitan_dir is None:
    raise FileNotFoundError(
        "Could not find synced 'torchtitan' repository. Sync your root project "
        "(e.g. google-pytorch) via `jupyter-sync sync` or set TORCHTITAN_DIR."
    )

# 2. Ensure TorchTitan requirements are installed into shared PYTHONUSERBASE
user_base = shared_mount_path / ".pydeps"
user_base.mkdir(parents=True, exist_ok=True)
tpu_trainer.configure_runtime_env(user_base=user_base, extra_pythonpath=torchtitan_dir)

required_mods = ("tyro", "grain", "tokamax", "spmd_types", "attn_gym", "torch_remat", "torchao", "transformers")
missing_mods = [m for m in required_mods if importlib.util.find_spec(m) is None]
if missing_mods:
    print(f"Installing TorchTitan requirements ({missing_mods}) from {torchtitan_dir} into {user_base}...")
    pip_env = {**os.environ, "PYTHONUSERBASE": str(user_base)}
    base_req_file = (
        torchtitan_dir / "requirements.txt"
        if (torchtitan_dir / "requirements.txt").exists()
        else torchtitan_dir / ".ci/docker/requirements.txt"
    )
    tpu_reqs = [
        line.strip()
        for line in (torchtitan_dir / "torchtitan/experiments/tpu/requirements.txt").read_text().splitlines()
        if line.strip() and not line.strip().startswith(("#", "jax==", "jaxlib==", "libtpu=="))
    ]
    for cmd in [
        [sys.executable, "-m", "pip", "install", "--user", "-q", *tpu_reqs],
        [
            sys.executable, "-m", "pip", "install", "--user", "-q",
            "-r", str(base_req_file),
            "-r", str(torchtitan_dir / ".ci/docker/requirements-flux.txt"),
        ],
        [
            sys.executable, "-m", "pip", "install", "--user", "-q", "--no-deps",
            "torchao==0.17.0", "typeguard>=4.0.0",
        ],
    ]:
        subprocess.check_call(cmd, env=pip_env)
    tpu_trainer.configure_runtime_env(user_base=user_base, extra_pythonpath=torchtitan_dir)
    print("TorchTitan dependencies installed.")
else:
    print(f"All TorchTitan dependencies are ready in '{user_base}'.")

_, repo_subpath, torchtitan_dir = tpu_trainer.resolve_shared_pvc_subpath(
    src_dir=torchtitan_dir,
    namespace=namespace,
    pvc_name=shared_pvc_name,
    shared_mount_path=str(shared_mount_path),
)
_, deps_subpath, _ = tpu_trainer.resolve_shared_pvc_subpath(
    src_dir=user_base,
    namespace=namespace,
    pvc_name=shared_pvc_name,
    shared_mount_path=str(shared_mount_path),
)

print(f"\nTarget Kubernetes Namespace : {namespace}")
print(f"Shared RWX PVC              : {pvc.metadata.name} (status={pvc.status.phase}, modes={pvc.spec.access_modes})")
print(f"Worker /workspace subPath   : {repo_subpath}")
print(f"Worker /workspace-deps      : {deps_subpath}")

## 2. Submit Multi-Host TorchTitan Training Job (`tpu_trainer.submit_multihost_training`)

We now launch a multi-host **Llama 3.2 1B (`llama3_1b`)** training job across **2 TPU v5e host nodes × 4 chips/host = 8 TPU chips** (`tpu-v5-8-multi-host`) using the generic `tpu_trainer.submit_multihost_training()` utility:
- `src_dir=torchtitan_dir` and `user_base=user_base` mount directly from `shared-workspace-rwx` at `/workspace` and `/workspace-deps` on every TPU worker pod.
- `main_script="torchtitan.experiments.tpu.train"` with `module=True` runs `torchrun --nproc_per_node=4 -m torchtitan.experiments.tpu.train` across all hosts.
- Kubeflow Trainer (`torch-distributed`), GKE's TPU webhook, and TorchTitan's `_maybe_init_distributed_on_gke()` automatically configure the 8-chip mesh (`2,4,1`) over high-speed ICI links.

In [ ]:
# Automatically use the same TPU container image as this Workspace pod (or TORCH_TPU_IMAGE override)
ws_pod = core_v1.read_namespaced_pod(name=os.environ["HOSTNAME"], namespace=namespace)
image = os.environ.get("TORCH_TPU_IMAGE") or ws_pod.spec.containers[0].image
print(f"Using TPU worker image: {image}\n")

job_name = tpu_trainer.submit_multihost_training(
    image=image,
    src_dir=torchtitan_dir,
    main_script="torchtitan.experiments.tpu.train",
    module=True,
    args=[
        "--module=torchtitan.experiments.tpu.llama3",
        "--config=llama3_1b",
        "--training.steps=5",
        "--training.max_context_length=128",
        "--training.dtype=bfloat16",
        "--training.mixed_precision_param=bfloat16",
        "--metrics.log_freq=1",
        "--metrics.enable_tensorboard",
    ],
    num_nodes=2,
    tpus_per_node=4,
    compute_class="tpu-v5-8-multi-host",
    pvc_name=shared_pvc_name,
    shared_mount_path=str(shared_mount_path),
    user_base=user_base,
)

## 3. Wait for Multi-Host TPU Slice Provisioning & Pod Scheduling

When `tpu_trainer.submit_multihost_training()` is called, GKE's Node Auto-Provisioner schedules a `2x4` TPU v5e slice (`tpu-v5-8-multi-host`). Wait for both TPU host worker pods (`node-0-0` and `node-0-1`) to reach `Running` status.

In [ ]:
tpu_trainer.wait_for_job_pods(job_name, num_nodes=2, timeout=900)

## 4. Stream Multi-Host TorchTitan Training Logs

Stream distributed training logs from all worker pods across the 2-host TPU slice (`world_size=8`). Observe how TorchTitan initializes `TORCH_TPU_TOPOLOGY=2,4,1` via `_maybe_init_distributed_on_gke()` and executes FSDP2 training steps across all 8 TPU chips.

In [ ]:
def make_torchtitan_pod_log_filter():
    """Surfaces TorchTitan rank-0 [titan] logs, TPU initialization, and errors across worker pods."""
    last_line_per_pod = {}
    keywords = (
        "Device type:",
        "Initializing TPU",
        "[titan]",
        "Training completed",
        "Process group destroyed",
        "Traceback",
        "RuntimeError",
        "ValueError",
        "Error:",
    )

    def _filter(pod_name: str, line: str):
        if "Training with config: {" in line:
            line = line.split("Training with config:")[0] + "Training with config: { ... }"
        if not any(k in line for k in keywords):
            return None
        clean = line.strip()
        if last_line_per_pod.get(pod_name) == clean:
            return None
        last_line_per_pod[pod_name] = clean
        return line

    return _filter


tpu_trainer.stream_job_logs(
    job_name,
    num_nodes=2,
    max_iterations=180,
    log_filter=make_torchtitan_pod_log_filter(),
)

## 5. Inspect Shared Outputs & Clean Up Multi-Host Resources

Because the multi-host worker pods mounted `/home/jovyan/shared/torchtitan` at `/workspace`, any TensorBoard logs, profile traces, or checkpoints written to `./outputs` during the distributed job are immediately accessible inside our Workspace pod without manual GCS downloads. Finally, we delete the `TrainJob` so GKE can scale down the multi-host TPU slice when idle.

In [ ]:
# Inspect TensorBoard event files written to the shared RWX volume by the multi-host job
tb_dir = torchtitan_dir / "outputs" / "tb"
if tb_dir.exists():
    event_files = sorted(tb_dir.rglob("events.out.tfevents.*"))
    print(f"Found {len(event_files)} TensorBoard event file(s) under {tb_dir}:")
    for ef in event_files[-5:]:
        print(f"  • {ef.relative_to(torchtitan_dir)} ({ef.stat().st_size} bytes)")

# Delete the TrainJob so the multi-host TPU slice can scale down cleanly
tpu_trainer.delete_training_job(job_name, delete_configmap=False)